In [ ]:
import collections

import json
import datasets
import numpy as np
import matplotlib.pyplot as plt
import tiktoken

tokenizer_gpt4o = tiktoken.encoding_for_model("gpt-4o")

In [ ]:

target_split = "train"  # "train", "test"
dataset = datasets.load_dataset("BAAI/TACO", split=target_split)

solution_counts = []
solution_lengths = []
broken_solutions_idxs = []
broken_input_output_idxs = []

raw_tags_counts = collections.defaultdict(int)
tags_counts = collections.defaultdict(int)
skill_types_counts = collections.defaultdict(int)

for sample_idx, sample in enumerate(dataset):
    solutions_str = sample["solutions"]
    try:
        solutions = json.loads(solutions_str)
    except:
        broken_solutions_idxs.append(sample_idx)
        continue
    if not solutions:
        broken_solutions_idxs.append(sample_idx)
        continue
    sample["solutions"] = solutions

    input_output_str = sample["input_output"]
    try:
        input_output = json.loads(input_output_str)
    except:
        broken_input_output_idxs.append(sample_idx)
        continue
    sample["input_output"] = input_output
    valid_inputs = "inputs" in input_output and len(input_output["inputs"]) > 0
    valid_outputs = "outputs" in input_output and len(input_output["outputs"]) > 0
    valid_pairs = len(input_output["inputs"]) == len(input_output["outputs"])
    if not (valid_inputs and valid_outputs and valid_pairs):
        broken_input_output_idxs.append(sample_idx)
        continue

    def _is_valid_type(val) -> bool:
        if isinstance(val, (list, set, dict)):
            if isinstance(val, dict):
                all_valid_keys = all([isinstance(k, (str, int, float)) for k in val.keys()])
                all_valid_subtypes = all([_is_valid_type(subval) for subval in val.values()])
                return all_valid_keys and all_valid_subtypes
            else:
                all_valid_subtypes = all([_is_valid_type(subval) for subval in val])
                return all_valid_subtypes
        else:
            return isinstance(val, (str, int, float)) or val is None

    if "fn_name" in input_output:
        assert isinstance(input_output["fn_name"], str) and input_output["fn_name"]
    # we just expect plain old data types for the args (will be converted to strings as needed)
    valid_input_types = all([_is_valid_type(i) for i in input_output["inputs"]])
    valid_output_types = all([_is_valid_type(o) for o in input_output["outputs"]])
    if not (valid_input_types and valid_output_types):
        broken_input_output_idxs.append(sample_idx)
        continue

    sample["raw_tags"] = eval(sample["raw_tags"])
    sample["tags"] = eval(sample["tags"])
    sample["skill_types"] = eval(sample["skill_types"])

    # Count occurrences of each type in raw_tags, tags, and skill_types
    for raw_tag in sample["raw_tags"]:
        raw_tags_counts[raw_tag] += 1
    for tag in sample["tags"]:
        tags_counts[tag] += 1
    for skill_type in sample["skill_types"]:
        skill_types_counts[skill_type] += 1

    solutions = sample.get("solutions", [])
    solution_counts.append(len(solutions))
    solutions_tokenized = [tokenizer_gpt4o.encode(s) for s in solutions]
    solution_lengths.extend([len(tokens) for tokens in solutions_tokenized])

broken_sample_idxs = list(set(broken_solutions_idxs + broken_input_output_idxs))

In [ ]:

print(f"Total sample count: {len(dataset)}")
print(f"Broken sample count: {len(broken_sample_idxs)}  (idxs={broken_sample_idxs})")
print(f"Total solution count (for all valid samples): {sum(solution_counts)}")

solution_avg_count = np.mean(solution_counts)
solution_min_count = np.min(solution_counts)
solution_max_count = np.max(solution_counts)
solution_std_count = np.std(solution_counts)
print(f"Average solution count per problem: {solution_avg_count:.2f}, min: {solution_min_count}, max: {solution_max_count}, std: {solution_std_count:.2f}")

solution_avg_length = np.mean(solution_lengths)
solution_min_length = np.min(solution_lengths)
solution_max_length = np.max(solution_lengths)
solution_std_length = np.std(solution_lengths)

print(f"Average token count per solution: {solution_avg_length:.2f}, min: {solution_min_length}, max: {solution_max_length}, std: {solution_std_length:.2f}")


In [ ]:
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.hist(solution_counts, bins=20, color="skyblue", edgecolor="black")
plt.yscale("log")
plt.title("Distribution of Solution Counts")
plt.xlabel("Number of Solutions")
plt.ylabel("Frequency (log scale)")

plt.subplot(1, 2, 2)
plt.hist(solution_lengths, bins=20, color="lightgreen", edgecolor="black")
plt.yscale("log")
plt.title("Distribution of Solution Lengths")
plt.xlabel("Solution Length (in tokens)")
plt.ylabel("Frequency (log scale)")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(18, 12))

def plot_percentage_distribution(data, title, xlabel, subplot_index):
    total = sum(data.values())
    percentages = {k: (v / total) * 100 for k, v in data.items()}
    sorted_percentages = dict(sorted(percentages.items(), key=lambda item: item[1], reverse=True))
    plt.subplot(2, 2, subplot_index)
    plt.bar(sorted_percentages.keys(), sorted_percentages.values(), color='gold', edgecolor='black')
    plt.xticks(rotation=90)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Percentage %")

plot_percentage_distribution(raw_tags_counts, "Percentage Distribution of Raw Tags", "Raw Tags", 1)
plot_percentage_distribution(tags_counts, "Percentage Distribution of Tags", "Tags", 2)
plot_percentage_distribution(skill_types_counts, "Percentage Distribution of Skill Types", "Skill Types", 3)

plt.tight_layout()
plt.show()

In [ ]:
# hardcoded values below found based on above cells
test_solutions_token_count = 69936 * 210.45  # valid count x avg token/solution
train_solutions_token_count = 1468713 * 159.05  # valid count x avg token/solution

total_solutions_token_count = test_solutions_token_count + train_solutions_token_count
print(total_solutions_token_count)

#expected_price_per_1M_tokens = 1.50  # USD, Anthropic Claude 3.7 Sonnet, with batch processing (2025-03-19)
expected_price_per_1M_tokens = 0.135  # USD, DeepSeek R1 with off-UTC-peak 75% discount (2025-03-19)
#expected_price_per_1M_tokens = 0.075  # USD, OpenAI GPT-4o-mini, with batch processing (2025-03-19)
expected_price_per_query = (total_solutions_token_count / 1e6) * expected_price_per_1M_tokens
print(f"Expected price per query: ${expected_price_per_query:.2f} (USD)")